In [8]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import numpy as np
import optuna
from sklearn.metrics import f1_score
from sklearn.metrics import accuracy_score, recall_score, precision_score, confusion_matrix, classification_report

In [2]:
df_2024 = pd.read_parquet("data_2024.parquet")

In [3]:
df_2024.head()

,_ASTHMS1,CHCKDNY2,_DRDXAR2,_EDUCAG,_INCOMG1,_AGE_G,_SEX,_BMI5,_RFDRHV9,_RFSMOK3,_TOTINDA,DIABETE4,year,DIABETE_BIN
0,3.0,0.0,1.0,2.0,9.0,6.0,1.0,2249.0,1.0,1.0,1.0,3.0,2024,0.0
1,3.0,0.0,1.0,4.0,7.0,6.0,0.0,2583.0,1.0,1.0,1.0,3.0,2024,0.0
2,3.0,0.0,1.0,3.0,9.0,5.0,0.0,2253.0,1.0,0.0,1.0,3.0,2024,0.0
3,3.0,0.0,1.0,4.0,4.0,6.0,0.0,2509.0,1.0,1.0,1.0,3.0,2024,0.0
4,3.0,0.0,0.0,3.0,2.0,4.0,0.0,1977.0,1.0,1.0,2.0,3.0,2024,0.0


In [4]:
target_col = "DIABETE_BIN"
df_2024 = df_2024.dropna()
X = df_2024.drop(columns=[target_col, "DIABETE4","year"])
y = df_2024[target_col]
X

,_ASTHMS1,CHCKDNY2,_DRDXAR2,_EDUCAG,_INCOMG1,_AGE_G,_SEX,_BMI5,_RFDRHV9,_RFSMOK3,_TOTINDA
0,3.0,0.0,1.0,2.0,9.0,6.0,1.0,2249.0,1.0,1.0,1.0
1,3.0,0.0,1.0,4.0,7.0,6.0,0.0,2583.0,1.0,1.0,1.0
2,3.0,0.0,1.0,3.0,9.0,5.0,0.0,2253.0,1.0,0.0,1.0
3,3.0,0.0,1.0,4.0,4.0,6.0,0.0,2509.0,1.0,1.0,1.0
4,3.0,0.0,0.0,3.0,2.0,4.0,0.0,1977.0,1.0,1.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...
453234,3.0,0.0,0.0,1.0,4.0,3.0,0.0,2466.0,1.0,1.0,1.0
453235,3.0,0.0,1.0,2.0,1.0,6.0,0.0,2986.0,1.0,1.0,2.0
453237,3.0,0.0,0.0,1.0,4.0,6.0,0.0,2066.0,0.0,0.0,2.0
453238,0.0,0.0,0.0,4.0,5.0,6.0,0.0,2437.0,1.0,1.0,2.0


## SMOTE

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


smote = SMOTE(random_state=42)

X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("Before SMOTE:", y_train.value_counts().to_dict())
print("After SMOTE:", y_train_res.value_counts().to_dict())


model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

model.fit(X_train_res, y_train_res)


y_prob = model.predict_proba(X_test)[:, 1]

y_pred = (y_prob >= 0.5).astype(int)

print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("Precision (class 1):", precision_score(y_test, y_pred, pos_label=1))
print("Recall (class 1):", recall_score(y_test, y_pred, pos_label=1))
print("F1 (class 1):", f1_score(y_test, y_pred, pos_label=1))

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

print("\nFull Report:\n", classification_report(y_test, y_pred))

print("\nMacro F1:", f1_score(y_test, y_pred, average="macro"))
print("Macro Recall:", recall_score(y_test, y_pred, average="macro"))

Before SMOTE: {0.0: 248805, 1.0: 51398}
After SMOTE: {0.0: 248805, 1.0: 248805}

Accuracy: 0.8298223874432052
Precision (class 1): 0.5106966538672518
Recall (class 1): 0.14490272373540855
F1 (class 1): 0.2257516973811833

Confusion Matrix:
 [[60417  1784]
 [10988  1862]]

Full Report:
               precision    recall  f1-score   support

         0.0       0.85      0.97      0.90     62201
         1.0       0.51      0.14      0.23     12850

    accuracy                           0.83     75051
   macro avg       0.68      0.56      0.57     75051
weighted avg       0.79      0.83      0.79     75051


Macro F1: 0.5650785940762779
Macro Recall: 0.5581107564112003


## SMOTE + OPTUNA

In [14]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

def objective(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 800),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "random_state": 42,
        "eval_metric": "logloss"
    }

    model = XGBClassifier(**params)
    model.fit(X_train_res, y_train_res)

    y_prob = model.predict_proba(X_test)[:, 1]

    best_f1 = 0
    for t in [0.2, 0.3, 0.4, 0.5]:
        y_pred = (y_prob >= t).astype(int)
        f1 = f1_score(y_test, y_pred, pos_label=1)
        best_f1 = max(best_f1, f1)

    return best_f1

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)

print("Best params:", study.best_params)
print("Best F1:", study.best_value)

[I 2026-04-19 23:27:35,310] A new study created in memory with name: no-name-64f5c05b-e5ff-437c-af0c-c7cc43638d3d
[I 2026-04-19 23:27:52,946] Trial 0 finished with value: 0.43467557587480216 and parameters: {'n_estimators': 380, 'max_depth': 6, 'learning_rate': 0.06971951347533656, 'subsample': 0.6251625498113422, 'colsample_bytree': 0.9704404914128109, 'min_child_weight': 7, 'gamma': 3.363578219819673}. Best is trial 0 with value: 0.43467557587480216.
[I 2026-04-19 23:28:12,514] Trial 1 finished with value: 0.43493813405265935 and parameters: {'n_estimators': 753, 'max_depth': 8, 'learning_rate': 0.1523316239608964, 'subsample': 0.7179049747122508, 'colsample_bytree': 0.8532301762366619, 'min_child_weight': 10, 'gamma': 4.401360939733563}. Best is trial 1 with value: 0.43493813405265935.
[I 2026-04-19 23:28:55,065] Trial 2 finished with value: 0.43615317618632876 and parameters: {'n_estimators': 625, 'max_depth': 8, 'learning_rate': 0.01342199458441444, 'subsample': 0.7749672566752963

Best params: {'n_estimators': 310, 'max_depth': 3, 'learning_rate': 0.10544802842427721, 'subsample': 0.6697850190170869, 'colsample_bytree': 0.9099241063557251, 'min_child_weight': 4, 'gamma': 1.39217260181068}
Best F1: 0.43776677479938536


In [15]:
best_params = study.best_params

model = XGBClassifier(**best_params)
model.fit(X_train_res, y_train_res)

y_prob = model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

In [16]:
from sklearn.metrics import classification_report, f1_score, recall_score

print(classification_report(y_test, y_pred))

print("Class 1 F1:", f1_score(y_test, y_pred, pos_label=1))
print("Class 1 Recall:", recall_score(y_test, y_pred, pos_label=1))

              precision    recall  f1-score   support

         0.0       0.85      0.97      0.90     62201
         1.0       0.51      0.15      0.24     12850

    accuracy                           0.83     75051
   macro avg       0.68      0.56      0.57     75051
weighted avg       0.79      0.83      0.79     75051

Class 1 F1: 0.2362063823441694
Class 1 Recall: 0.15408560311284047
